In [1]:
import sys
import os
!{sys.executable} -m pip install -q torch numpy matplotlib plotly ipywidgets tqdm sentence_transformers scikit-learn spacy dash flask waitress


[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
import numpy as np
import pickle
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from IPython.display import display
import ipywidgets as widgets
import math
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer
import re
from sklearn.preprocessing import MinMaxScaler
import spacy
from IPython.display import display
from dash import Dash, html, dcc, Input, Output, State
from flask import Flask, render_template_string, request, jsonify
import json

def plot(text_segments, e_matrix, n_segments, topics, lengths, n_c, chapter_names):
    x_vals = np.arange(n_segments)
    markers = ['circle', 'square', 'triangle-up', 'diamond', 'triangle-down',
            'star', 'pentagon', 'hexagon', 'x', 'cross', 'diamond-open']

    fig = go.Figure()

    for t_idx, topic_key in enumerate(topics.keys()):
        fig.add_trace(go.Scatter(
            x=x_vals,
            y=e_matrix[:, t_idx],
            mode='lines+markers',
            name=topic_key,
            marker=dict(symbol=markers[t_idx % len(markers)], size=8),
            hovertemplate=(
                "Index: %{x}<br>"
                "Value e(t): %{y:.2f}<br>"
                "Topic: " + topic_key + "<extra></extra>"
            )
        ))

    previous = 0
    min_y = float(np.min(e_matrix))
    max_y = float(np.max(e_matrix))
    for seg in lengths[:-1]:
        boundary = seg + previous
        fig.add_shape(
            type="line",
            x0=boundary, y0=min_y,
            x1=boundary, y1=max_y,
            line=dict(color="gray", dash="dash")
        )
        previous += seg

    fig.update_layout(
        title=f"Chapter {n_c} - {chapter_names[n_c][1:]}",
        xaxis_title="Segment index",
        yaxis_title="Thematic force e(t)",
        template="plotly_white",
        legend=dict(x=1.05, y=1),
        width=1200,
        height=700
    )

    mode_selector = widgets.ToggleButtons(
        options=['Single Segment', 'Segment Range'],
        description='Mode:',
        style={'description_width': 'initial'}
    )

    single_slider = widgets.IntSlider(
        value=0, min=0, max=n_segments - 1, description='Select Segment:',
        style={'description_width': 'initial'}, layout=widgets.Layout(width='1200px')
    )

    range_slider = widgets.IntRangeSlider(
        value=[0, 1], min=0, max=n_segments - 1, description='Select Segment Range:',
        style={'description_width': 'initial'}, layout=widgets.Layout(width='1200px'),
        visible=False
    )

    output_widget = widgets.HTML(
        layout=widgets.Layout(
            border='1px solid gray', height='300px',
            width='1200px', overflow_y='scroll', padding='10px'
        )
    )

    def update_text(*args):
        if mode_selector.value == 'Single Segment':
            idx = single_slider.value
            content = f"<h4>Segment {idx}</h4><p>{text_segments[idx]}</p>"
        else:
            start, end = range_slider.value
            content = ""
            for idx in range(start, end + 1):
                content += f"<h4>Segment {idx}</h4><p>{text_segments[idx]}</p><hr>"
        output_widget.value = content

    def toggle_mode(change):
        if change['new'] == 'Single Segment':
            single_slider.layout.display = 'block'
            range_slider.layout.display = 'none'
        else:
            single_slider.layout.display = 'none'
            range_slider.layout.display = 'block'
        update_text()

    mode_selector.observe(toggle_mode, names='value')
    single_slider.observe(update_text, names='value')
    range_slider.observe(update_text, names='value')

    single_slider.layout.display = 'block'
    range_slider.layout.display = 'none'
    update_text()

    
    display(fig)
    display(mode_selector, single_slider, range_slider, output_widget)
    

In [3]:
class LexicalTopicStrength:
    
    def __init__(self, topics, preprocess=True, tau=1, perfect_match_reward = 10, similarity_match_reward=2):
        self.topics = topics
        self.tau = tau
        self.vocab = set()
        self.preprocess = preprocess
        self.perfect_match_reward = perfect_match_reward
        self.similarity_match_reward = similarity_match_reward
        if isinstance(self.topics, dict):
            self.order_keys = sorted(self.topics.keys())
        
        if preprocess: 
            print('Preprocessing the topics...')
            self.nlp = spacy.load("it_core_news_sm")
            if isinstance(self.topics, dict):
                self.topics = self._preprocess_dict(self.nlp, self.topics)
            else:
                self.topics = self._preprocess_list(self.nlp, self.topics)
        
        
        self._build_vocabulary(self.topics)

        
    def _preprocess_text(self, nlp, text):
        doc = nlp(text)
        
        custom_stopwords = {
            "il", "lo", "la", "l'", "un", "una", "uno", 
            "dei", "degli", "delle", "del", "della", 
            "dell'", "dallo", "dalla", "dagli", "dalle",'all',"all'", 'di','da',
            'in','con','su','per','tra','fra','e','o','ed','ma','ecc','ecc.' }

        tokens = []
        for token in doc:
            if token.is_stop or token.is_punct or token.is_space:
                continue
            if token.lemma_.lower() in custom_stopwords:
                continue
            
            tokens.append(token.lemma_.lower())
        
        return " ".join(tokens)

    def _preprocess_dict(self, nlp, topics):
        processed_dict = {}
        for key in self.order_keys:
            text_list = self.topics[key]
            processed_list = [self._preprocess_text(nlp, text) for text in text_list]
            processed_dict[key] = processed_list
        return processed_dict
    
    def _preprocess_list(self, nlp, topics):
        preprocessed = []
        for text in topics:
            text_prep = self._preprocess_text(nlp, text)
            preprocessed.append(text_prep)
        return preprocessed
    
    def get_vocab(self):
        return self.vocab
    
    def _build_vocabulary(self, topics):
        if isinstance(topics, dict):
            for tlist in topics.values():
                for phrase in tlist:
                    for w in re.findall(r'\w+', phrase.lower()):
                        self.vocab.add(w)
        else:
            for phrase in topics:
                for w in re.findall(r'\w+', phrase.lower()):
                    self.vocab.add(w)
                    
    def _compute_idf(self, paragraphs):
        N = len(paragraphs)
        if self.preprocess:
            paragraph_sets = [self._preprocess_text(self.nlp ,p) for p in paragraphs]
        else:
            paragraph_sets = [set(re.findall(r'\w+', p.lower())) for p in paragraphs]
        idf = {}
        for w in self.vocab:
            df = sum(1 for s in paragraph_sets if w in s)
            idf[w] = math.log((N + 1) / (df + 1)) + 1
        return idf

    def softmax(self, matrix, t):
        matrix_stable = matrix - np.max(matrix, axis=1, keepdims=True)
        exp_matrix = np.exp(matrix_stable / t)
        softmax_matrix = exp_matrix / np.sum(exp_matrix, axis=1, keepdims=True)
        return softmax_matrix
   
    def compute_lexical_scores(self, paragraphs, normalization=None):
        if isinstance(self.topics, dict):
            idf = self._compute_idf(paragraphs)
            scores = np.zeros((len(paragraphs), len(self.topics)))
            paragraphs = self._preprocess_list(self.nlp, paragraphs)
            for i, para in enumerate(paragraphs):
                words = re.findall(r'\w+', para.lower())
                freq = Counter(words)
                for k, tname in enumerate(self.order_keys):
                    val = 0.0
                    total_topic_words = 0 
                    for phrase in self.topics[tname]:
                        
                            
                        if para.strip().lower() == phrase.strip().lower():
                            val += self.perfect_match_reward
                        words_in_phrase = re.findall(r'\w+', phrase.lower())
                        total_topic_words += len(words_in_phrase)
                        for w in words_in_phrase:
                            if w in freq:
                                val += freq[w] * idf.get(w, 0.0)
                                
                    val /= np.log2(total_topic_words)

                    scores[i, k] = val #torch.sigmoid(torch.tensor(val)).numpy()
                
            if normalization.lower()=='softmax':
                row_sums = scores.max(axis=1)
                scores = scores / row_sums[:, np.newaxis]
            elif normalization.lower()=='row':
                scores = self.softmax(scores, self.tau)
            elif normalization.lower()=='log':
                scores = np.log2(scores+1)                
                scores = min_max(scores, x=1)
            return scores
        else:
            idf = self._compute_idf(paragraphs)
            scores = []
            
            for i, para in enumerate(paragraphs):
                words = re.findall(r'\w+', para.lower())
                freq = Counter(words)
                val = 0.0
                for phrase in self.topics:
                    for w in re.findall(r'\w+', phrase.lower()):
                    
                        if w in freq:
                            val += freq[w] * idf.get(w, 0.0)
                
                
                scores.append(val)
                
            scores = np.array(scores)
            
            if normalization.lower()=='log':
                scores = np.log2(scores+1)               
                scores = MinMaxScaler(feature_range=(0, 1)).fit_transform(scores.reshape(-1, 1))
            return scores#/scores.sum()
            
            


class SemanticTopicStrength:
    
    def __init__(self, topics, model_name, device="cuda", dim_size=3, bandwidth=1, dr_method='PCA', batch_size=16, use_kde=True, tau=0.5):
        self.topics = topics
        self.model_name = model_name
        self.device = device
        self.tau = tau
        self.model = SentenceTransformer(self.model_name, device=device)
        self.batch_size = batch_size
        if isinstance(self.topics, dict):
            self.order_keys = sorted(self.topics.keys())
        
        self.use_kde = use_kde
        if self.use_kde:
            self.dr_method = dr_method.upper() if dr_method else None
            self.dim_size = dim_size
            self.bandwidth = bandwidth
            self.reducers = {}
            self.kdes = {}
            self._build_topic_kdes()
        else:
            if isinstance(self.topics, dict):
                self.topic_embeddings, self.topic_length = self._compute_topic_embeddings(self.batch_size)
                
            else:
                self.topic_embeddings = self.compute_block_embeddings()
                self.topic_length = 1
            
    def _encode_batch(self, sentences):
        with torch.no_grad():
            embs = self.model.encode(sentences, convert_to_numpy=True)
        return embs
    
    def split_into_chunks(self, text, max_chunk_chars=500):
        sentences = re.split(r'(?<=\.)\s+', text.strip())
        chunks = []
        current_chunk = ""
        
        for sentence in sentences:
            if current_chunk:
                if len(current_chunk) + len(sentence) + 1 <= max_chunk_chars:
                    current_chunk += " " + sentence
                else:
                    chunks.append(current_chunk.strip())
                    current_chunk = sentence
            else:
                current_chunk = sentence
        
        if current_chunk:
            chunks.append(current_chunk.strip())
            
        return chunks

    def compute_block_embeddings(self,  max_chunk_chars=500):
        
        all_topic_embeddings = []
        
        for topic in self.topics: # list of par
            
            chunks = self.split_into_chunks(topic, max_chunk_chars)
            chunk_embeddings = self.model.encode(chunks)
            topic_avg_embedding = np.mean(chunk_embeddings, axis=0)
            all_topic_embeddings.append(topic_avg_embedding)
        overall_avg_embedding = np.mean(all_topic_embeddings, axis=0)
        return overall_avg_embedding
    
    def _compute_topic_embeddings(self, batch_size=16):
        all_embs = []
        topic_length = []
        for tname in self.order_keys:
            phrases = self.topics[tname]
            topic_length.append(len(phrases))
            dataset = SentenceDataset(phrases)
            loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
            for batch in loader:
                emb = self._encode_batch(batch)
                all_embs.append(emb)
        emb_matrix = np.concatenate(all_embs, axis=0)
        return emb_matrix, topic_length
    
    def _build_topic_kdes(self):
        for topic_key in self.order_keys:
            topic_examples = self.topics[topic_key]
            if not topic_examples:
                self.reducers[topic_key] = None
                self.kdes[topic_key] = None
                continue
            dataset = SentenceDataset(topic_examples)
            loader = DataLoader(dataset, batch_size=self.batch_size)
            all_emb = []
            for batch in loader:
                emb = self._encode_batch(batch)
                all_emb.append(emb)
            emb_matrix = np.concatenate(all_emb, axis=0)
            centroid = np.mean(emb_matrix, axis=0, keepdims=True)
            emb_matrix = np.vstack([emb_matrix, centroid])
            if self.dr_method == "PCA":
                real_dim = min(self.dim_size, emb_matrix.shape[0], emb_matrix.shape[1])
                pca = PCA(n_components=real_dim)
                red_data = pca.fit_transform(emb_matrix)
                self.reducers[topic_key] = pca
            elif self.dr_method == "UMAP" and umap is not None:
                real_dim = min(self.dim_size, emb_matrix.shape[0], emb_matrix.shape[1])
                reducer = umap.UMAP(n_components=real_dim, random_state=42,
                                    metric='cosine', init='pca')
                red_data = reducer.fit_transform(emb_matrix)
                self.reducers[topic_key] = reducer
            else:
                red_data = emb_matrix
                self.reducers[topic_key] = None

            kde_model = KernelDensity(bandwidth=self.bandwidth, kernel='gaussian')
            kde_model.fit(red_data)
            self.kdes[topic_key] = kde_model
    
    def compute_centroids(self, keyword_embedding, topic_length):
        
        topic_vectors = []
        idx = 0
        for length in topic_length:

            vecs = keyword_embedding[idx : idx + length]
            idx += length
            if vecs.shape[0] == 1:
            
                topic_vector = vecs[0]
            else:
                centroid = np.mean(vecs, axis=0) 
                topic_vector = centroid
            topic_vectors.append(topic_vector)
        topic_vectors = np.vstack(topic_vectors) 
        return topic_vectors
    
    
    
    def compute_paragraph_embeddings(self, paragraphs):
        return np.array(self.model.encode(paragraphs))

    def cosine_similarity_matrix(self, paragraph_embeddings, topic_vectors):
        topic_norms = np.linalg.norm(topic_vectors, axis=1, keepdims=True)
        topic_vectors_norm = topic_vectors / topic_norms
        para_norms = np.linalg.norm(paragraph_embeddings, axis=1, keepdims=True)
        para_vectors_norm = paragraph_embeddings / para_norms
        

        similarity_matrix = np.dot(para_vectors_norm, topic_vectors_norm.T)
        #similarity_matrix = (similarity_matrix + 1.0) / 2.0
        
        return similarity_matrix
    
    def softmax(self, matrix, t):
        matrix_stable = matrix - np.max(matrix, axis=1, keepdims=True)
        exp_matrix = np.exp(matrix_stable / t)
        softmax_matrix = exp_matrix / np.sum(exp_matrix, axis=1, keepdims=True)
        return softmax_matrix
    
    
    def cosine_similarity_matrix(self, A_np, B_np):
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        A = torch.tensor(A_np, device=device, dtype=torch.float32)
        B = torch.tensor(B_np, device=device, dtype=torch.float32)
        A = A / A.norm(dim=1, keepdim=True)
        B = B / B.norm(dim=1, keepdim=True)
        sim = torch.mm(A, B.t())
        return sim.cpu().numpy()
    


    def combine_blocks(self,
            M: np.ndarray,
            block_boundaries: list,
            transform='relu',
            penalty='log'
        ):
        num_rows = M.shape[0]
        num_blocks = len(block_boundaries)
        
        if transform == 'relu':
            def f(x): return np.maximum(x, 0.0)
        elif transform == 'linear':
            def f(x): return (x + 1.) / 2.
        elif transform == 'identity':
            def f(x): return x
        else:
            raise ValueError("transform non valido!")
        
        out = np.zeros((num_rows, num_blocks))
        idx = 0
        
        for j, length in enumerate(block_boundaries):
            K_j = length
            start = idx
            end = length + idx
            idx += length

            if penalty == 'none':
                w_j = 1.0
            elif penalty == 'inverse':
                w_j = 1.0 / K_j
            elif penalty == 'inverse_sqrt':
                w_j = 1.0 / np.sqrt(K_j)
            elif penalty == 'log':
                w_j = 1.0 / np.log(K_j + 1.0)
            elif penalty == 'custom':
                alpha = 0.75
                w_j = 1.0 / (K_j ** alpha)
            else:
                raise ValueError("penalty non valido!")
            
            block_slice = M[:, start:end]
            block_slice_transformed = f(block_slice)
            block_mean = block_slice_transformed.mean(axis=1)  
            out[:, j] = block_mean * w_j
        
        return out

    
    def compute_semantic_scores(self, paragraphs, normalization='none'):
        if not self.use_kde:
            if isinstance(self.topics, dict):
                paragraph_embeddings = self.compute_paragraph_embeddings(paragraphs)
                #topic_vectors = self.model.similarity(paragraph_embeddings, self.topic_embeddings)
                topic_vectors = self.cosine_similarity_matrix(paragraph_embeddings, self.topic_embeddings)
                scores = self.combine_blocks(topic_vectors, self.topic_length, transform='identity', penalty='none')

                if normalization.lower()=='row':
                    row_sums = scores.sum(axis=1)
                    scores = scores / row_sums[:, np.newaxis]
                elif normalization.lower()=='softmax':
                    scores = self.softmax(scores, t=0.1)
                elif normalization.lower()=='log':
                    scores = np.log2(scores+1)                
                    scores = min_max(scores, x=1)
                
                return scores
            else:
                paragraph_embeddings = self.compute_paragraph_embeddings(paragraphs)
                scores = self.cosine_similarity_matrix(paragraph_embeddings, self.topic_embeddings.reshape(1, -1))
                if normalization.lower()=='log':
                    scores = np.log2(scores+1)                
                    scores = MinMaxScaler(feature_range=(0, 1)).fit_transform(scores.reshape(-1, 1))
                return scores
        else:
            
            paragraph_embeddings = self.compute_paragraph_embeddings(paragraphs)
            kdes_all = []
            for topic_key, sentence in self.topics.items():
                reducer = self.reducers[topic_key]
                kde = self.kdes[topic_key]
                
                para_red = reducer.transform(paragraph_embeddings)
                kde_scores = kde.score_samples(para_red)
                kdes_all.append(np.exp(kde_scores))
            kdes_all = np.array(kdes_all)
            scores = np.column_stack(kdes_all)
            if normalization.lower()=='softmax':
                scores = self.softmax(scores, t=self.tau)
            elif normalization.lower()=='log':
                scores = np.log2(scores+1)                
                scores = MinMaxScaler(feature_range=(0, 1)).fit_transform(scores.reshape(-1, 1))
            return scores
        

class ThematicStrength:
    def __init__(self, topics, model_name, device="cuda", use_kde=False, batch_size=16, tau_lexical=10, tau_semantic=0.5, adaptive_combination=True, normalization='log'):
        self.topics = topics
        self.model_name = model_name
        self.device = device
        self.lexical_strength = LexicalTopicStrength(topics, preprocess=True, tau=tau_lexical)
        self.semantic_strength = SemanticTopicStrength(topics, model_name, device, use_kde=use_kde, batch_size=batch_size, tau=tau_semantic)
        self.model = self.semantic_strength.model
        self.sentence_embeddings = None
        self.adaptive_combination = adaptive_combination
        self.normalization = normalization
        
            
    
    def global_confidence(self, vec):
        sorted_vals = np.sort(vec)[::-1]  
        top1 = sorted_vals[0]
        if len(sorted_vals) > 1:
            top2 = sorted_vals[1]
        else:
            top2 = 0.0
        return max(0.0, top1 - top2)  
    def combine_adaptive(self, lex_vector, sem_vector, gamma=2.0, eps=1e-9):
        c_lex = self.global_confidence(lex_vector)
        c_sem = self.global_confidence(sem_vector)
        c0 = c_lex / (c_lex + c_sem + eps)
        
        diff = np.abs(lex_vector - sem_vector)
        delta = np.minimum(1.0, gamma * diff)
        
        w = 0.5*(1 - delta) + c0*delta
        
        combined = w*lex_vector + (1 - w)*sem_vector
        
        return combined, w 
    
    
    def compute_mixing_weights(self, lex_dist, sem_dist, alpha=5.0, conflict_threshold=0.1):

        n = lex_dist.shape[0]
        w = np.zeros(n)
        for i in range(n):
            eps = 1e-9
            Hl = -np.sum(lex_dist[i] * np.log(lex_dist[i] + eps))
            Hs = -np.sum(sem_dist[i] * np.log(sem_dist[i] + eps))
            delta = Hs - Hl

            top_lex = np.argmax(lex_dist[i])
            top_sem = np.argmax(sem_dist[i])

            if top_lex == top_sem:
                w[i] = 1 / (1 + math.exp(-alpha * delta))
            else:
                if abs(delta) > conflict_threshold:
                    w[i] = 1.0 if Hl < Hs else 0.0
                else:
                    w[i] = 1 / (1 + math.exp(-2 * alpha * delta))
        return w

    def combine_distributions(self, lex_dist, sem_dist, mixw):
        cd = (mixw[:,None]*lex_dist + (1-mixw[:,None])*sem_dist)
        cd = cd / (np.sum(cd, axis=1, keepdims=True) + 1e-9)
        return cd
    
    
    def compute_thematic_strength(self, paragraphs, return_dict=False, ni=0.5, gamma=2.0):
    
        lex_dist = self.lexical_strength.compute_lexical_scores(paragraphs, normalization=self.normalization)
        sem_dist = self.semantic_strength.compute_semantic_scores(paragraphs, normalization=self.normalization)
        if self.adaptive_combination:
            combined = np.zeros_like(lex_dist)
            for i in range(len(paragraphs)):
                combined[i], _ = self.combine_adaptive(lex_dist[i], sem_dist[i], gamma=gamma)
        else:    
            combined = (1-ni) * lex_dist + ni*sem_dist
        
        if return_dict:
            topic_list= list(self.topics.keys())
            out = {}
            for i, row in enumerate(combined):
                tmp = {}
                for k, tname in enumerate(topic_list):
                    tmp[tname] = float(row[k])
                out[i] = tmp
            return out
        return combined
    
        
        

class TopicFlowEquation:

    def __init__(self, topics, paragraphs, chapter_length= None, doc=None, cultural_strength=None, chapter_inter_length=None, sentence_embedding=None, model_name='paraphrase-multilingual-MiniLM-L12-v2', batch_size=16, device='cuda'):
        
        self.embedder = ThematicStrength(topics=topics, model_name=model_name, device=device, use_kde=False, batch_size=batch_size, 
                                         adaptive_combination=True)
        
        print('Initializing thematic function...')
        self.strength = self.embedder.compute_thematic_strength(paragraphs, ni=0.5)
        self.model_name = model_name
        self.cultural = None
        self.chapter_length = chapter_length
        self.paragraphs = paragraphs
        self.chapter_inter_length = chapter_inter_length
        if cultural_strength is not None:
            self.cultural_strength = cultural_strength
            self.mean_strength = np.mean(self.strength, axis=0)

        self.topics = topics

        self.num_topics = len(topics)
        self.sentence_embeddings = sentence_embedding
        
        


    def cosine_similarity(self, a, b):
        norm_a = np.linalg.norm(a) + 1e-12
        norm_b = np.linalg.norm(b) + 1e-12
        return np.dot(a, b) / (norm_a * norm_b)
    
    def init(self):
        self.c_strength = np.copy(self.strength)
    
    def get_strength(self, segment_idx, topic_idx):
        return self.strength[segment_idx][topic_idx]
    
    def evolve_strength(self, segment_idx, topic_idx,  u=1.0, nu = 0.2, D=0.1):
        if segment_idx == 0:
            return self.c_strength[0][topic_idx] 
    
        prev_strength = self.c_strength[segment_idx - 1][topic_idx]
        new_strength = prev_strength

        if segment_idx >= 2:
            recent_trend = self.c_strength[segment_idx - 1][topic_idx] - self.c_strength[segment_idx - 2][topic_idx]
            new_strength += u * recent_trend  # if u=1, essentially carries full last change forward

        
        diffusion_term = 0.0
        norm_factor = 0.0
        emb_current = self.sentence_embeddings[segment_idx]
        if segment_idx > 1:
            emb_prev = self.sentence_embeddings[segment_idx - 1]
            sim_prev = self.cosine_similarity(emb_current, emb_prev)
            diffusion_term += sim_prev * (self.c_strength[segment_idx - 1][topic_idx] - prev_strength)
            norm_factor += sim_prev

        if segment_idx < len(self.sentence_embeddings) - 1:
            emb_next = self.sentence_embeddings[segment_idx + 1]
            sim_next = self.cosine_similarity(emb_current, emb_next)
            diffusion_term += sim_next * (self.c_strength[segment_idx + 1][topic_idx] - prev_strength) # <-- attenzione qui!
            norm_factor += sim_next

        if norm_factor > 0:
            diffusion_term /= norm_factor
            new_strength += D * diffusion_term
        change = new_strength - prev_strength
        new_strength -= nu * change  
        g_cultural = 0.0
        if self.cultural:
            g_cultural = self.cultural_strength[segment_idx]
        new_strength += 1e-1 * g_cultural
        if new_strength < 0:
            new_strength = 0  

        self.c_strength[segment_idx][topic_idx] = new_strength
        return new_strength

In [5]:
import pathlib
import io

pathlib.PosixPath = pathlib.WindowsPath
def cpu_load(*args, **kwargs):
    return torch.load(*args, map_location='cpu', **kwargs)

torch.serialization.load = cpu_load


class CPU_Unpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module == 'torch.storage' and name == '_load_from_bytes':
            return lambda b: torch.load(io.BytesIO(b), map_location='cpu')
        else:
            return super().find_class(module, name)

with open("ecoflow.pkl", "rb") as f:
    flow_eq = CPU_Unpickler(f).load()
    
flow_eq.init()
    

In [6]:

flow_eq.init()
chapter_names = {1:'_KETEL', 2:'_HOKMAH', 3:'_BINAH', 4:'_HESED'}
n_c = 1
topics = flow_eq.topics
paragraphs = flow_eq.paragraphs
chapter_length = flow_eq.chapter_length
doc = flow_eq.doc
assert isinstance(doc, list)
doc = doc[0]
cultural_strength = flow_eq.cultural_strength
chapter_inter_length = flow_eq.chapter_inter_length
sentence_embedding = flow_eq.sentence_embeddings


this_chap = chapter_length[n_c]
text_segments = paragraphs[:this_chap]
lengths = chapter_inter_length[n_c]

n_segments = len(text_segments)
n_topics = len(topics)

print(f'Simulating flow on chapter {chapter_names[n_c]}')
e_matrix = np.zeros((n_segments, n_topics))
for t_idx, topic_key in enumerate(topics.keys()):
    for i in range(n_segments):
        e_next = flow_eq.evolve_strength(i, t_idx,  nu = 0.15, D=0.15)
        e_matrix[i, t_idx] = e_next

plot(text_segments, e_matrix, n_segments, topics, lengths, n_c, chapter_names)

        




Simulating flow on chapter _KETEL


ToggleButtons(description='Mode:', options=('Single Segment', 'Segment Range'), style=ToggleButtonsStyle(descr…

IntSlider(value=0, description='Select Segment:', layout=Layout(display='block', width='1200px'), max=41, styl…

IntRangeSlider(value=(0, 1), description='Select Segment Range:', layout=Layout(display='none', width='1200px'…

HTML(value="<h4>Segment 0</h4><p>Fu allora che vidi il Pendolo. La sfera, mobile all'estremità di un lungo fil…

In [9]:
import numpy as np
import plotly.graph_objs as go
from flask import Flask, render_template_string, request, jsonify
import json

app = Flask(__name__)

data = {}


def generate_e_matrix(flow_eq, chapter_length, n_c, u, nu, D):
    this_chap = chapter_length[n_c]
    n_segments = len(flow_eq.paragraphs[:this_chap])
    n_topics = len(flow_eq.topics)

    e_matrix = np.zeros((n_segments, n_topics))
    for t_idx, topic_key in enumerate(flow_eq.topics.keys()):
        for i in range(n_segments):
            e_matrix[i, t_idx] = flow_eq.evolve_strength(i, t_idx, nu=nu, u=u, D=D)
    return e_matrix

def init_data(flow_eq, chapter_names, chapter_length, chapter_inter_length):
    global data
    data = {
        'flow_eq': flow_eq,
        'chapter_names': chapter_names,
        'chapter_length': chapter_length,
        'chapter_inter_length': chapter_inter_length,
        'chapters': {}
    }
    for n_c in chapter_names:
        data['chapters'][n_c] = {
            'nu': 0.15,
            'D': 0.15,
            'u': 1.0,
            'e_matrix': generate_e_matrix(flow_eq, chapter_length, n_c, 1.0, 0.15, 0.15)
        }

@app.route('/')
def index():
    chapter_options = [{'id': c, 'name': data['chapter_names'][c]} for c in data['chapter_names']]
    return render_template_string("""
    <!DOCTYPE html>
    <html>
    <head>
        <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    </head>
    <body>
        <select id="chapterSelect">
            {% for c in chapter_options %}
                <option value="{{c.id}}">{{c.name}}</option>
            {% endfor %}
        </select>
        u: <input type="number" id="u" value="1.0" step="0.1">
        nu: <input type="number" id="nu" value="0.15" step="0.01">
        D: <input type="number" id="D" value="0.15" step="0.01">
        <button onclick="loadChapter()">Load Chapter</button>
        <div id='graph' style='width:1200px; height:700px;'></div>
        <div>
            Select Segment Range: 
            <input type="number" id="start" min="0" value="0">
            <input type="number" id="end" min="0" value="10">
            <button onclick="loadSegments()">Load Segments</button>
            <label><input type="checkbox" id="combine"> Combine Segments</label>
        </div>
        <div id="segments" style='width:1200px; height:300px; overflow-y:scroll; border:1px solid #ccc; padding:10px;'></div>
        <script>
            let currentChapter = 1;
            function loadChapter() {
                const chapter = document.getElementById('chapterSelect').value;
                currentChapter = chapter;
                const nu = document.getElementById('nu').value;
                const D = document.getElementById('D').value;
                const u = document.getElementById('u').value;
                fetch(`/chapter/${chapter}?nu=${nu}&D=${D}&u=${u}`).then(res=>res.json()).then(plot_data=>{
                    Plotly.newPlot('graph', plot_data.data, plot_data.layout);
                    document.getElementById('end').value = plot_data.data[0].x.length - 1;
                });
            }

            function loadSegments(){
                var start = document.getElementById('start').value;
                var end = document.getElementById('end').value;
                var combine = document.getElementById('combine').checked;
                fetch(`/segments/${currentChapter}?start=${start}&end=${end}&combine=${combine}`)
                .then(res => res.json())
                .then(d => {
                    document.getElementById('segments').innerHTML = d.content;
                });
            }

            document.getElementById('graph').on('plotly_click', function(data){
                var idx = data.points[0].x;
                fetch(`/segment/${currentChapter}/${idx}`)
                .then(res => res.json())
                .then(d => {
                    document.getElementById('segments').innerHTML = `<h4>Segment ${idx}</h4><p>${d.text}</p>`;
                });
            });

            window.onload = loadChapter;
        </script>
    </body>
    </html>
    """, chapter_options=chapter_options)

@app.route('/chapter/<int:n_c>')
def load_chapter(n_c):
    nu = float(request.args.get('nu', 0.15))
    D = float(request.args.get('D', 0.15))
    u = float(request.args.get('u', 1.0))
    data['flow_eq'].init()
    chapter_data = data['chapters'][n_c]
    chapter_data['nu'] = nu
    chapter_data['D'] = D
    chapter_data['u'] = u
    chapter_data['e_matrix'] = generate_e_matrix(data['flow_eq'], data['chapter_length'], n_c, u, nu, D)

    e_matrix = chapter_data['e_matrix']
    segments = list(range(len(e_matrix)))
    traces = []

    for t_idx, topic_key in enumerate(data['flow_eq'].topics.keys()):
        traces.append(go.Scatter(x=segments, y=list(e_matrix[:, t_idx]), mode='lines+markers', name=topic_key, marker=dict(size=8)))

    fig = go.Figure(data=traces)
    fig.update_layout(template='plotly_white', width=1200, height=700, title=f"Chapter {n_c} - {data['chapter_names'][n_c]}")
    return jsonify(json.loads(fig.to_json()))

@app.route('/segment/<int:n_c>/<int:idx>')
def get_segment(n_c, idx):
    text = data['flow_eq'].paragraphs[idx]
    return jsonify({'text': text})

@app.route('/segments/<int:n_c>')
def get_segments(n_c):
    start = int(request.args.get('start', 0))
    end = int(request.args.get('end', start))
    combine = request.args.get('combine', 'false').lower() == 'true'
    texts = [data['flow_eq'].paragraphs[i] for i in range(start, end + 1)]
    content = "<hr>".join(texts) if combine else ''.join(f"<h4>Segment {i}</h4><p>{texts[i-start]}</p><hr>" for i in range(start, end+1))
    return jsonify({'content': content})


init_data(flow_eq, chapter_names, chapter_length, chapter_inter_length)

app.run(debug=False, port=5050, use_reloader=False)


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5050
Press CTRL+C to quit
127.0.0.1 - - [01/Mar/2025 16:05:49] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [01/Mar/2025 16:05:52] "GET /chapter/1?nu=0.15&D=0.15&u=1.0 HTTP/1.1" 200 -
127.0.0.1 - - [01/Mar/2025 16:05:54] "GET /segments/1?start=0&end=41&combine=false HTTP/1.1" 200 -
127.0.0.1 - - [01/Mar/2025 16:05:56] "GET /segments/1?start=0&end=41&combine=true HTTP/1.1" 200 -
127.0.0.1 - - [01/Mar/2025 16:06:05] "GET /chapter/2?nu=0.15&D=0.15&u=1.0 HTTP/1.1" 200 -
127.0.0.1 - - [01/Mar/2025 16:06:10] "GET /segments/2?start=0&end=73&combine=true HTTP/1.1" 200 -
